# 02 - Shared feature function on the real data: memory, parity, cache (S1-09)

**Owner:** Bibek. **Spec:** `docs/FEATURE_SPEC.md` (schema `0.2.0`, 12 features). **Issue:** S1-09.

**What this notebook proves, in order**

1. The shared function (`build_feature_table`) runs over all 6,362,620 real rows, and what it costs (time, peak memory) on the first 1M rows and on the full pass.
2. The batch features for the replay pool are **exactly** what the Flow A worker would compute: state primed at the replay boundary (recipient accounts of the replay pool only, D6), then the worker sequence `compute_features` -> `update_dest_state`, with state round-tripped through JSON in the planned Redis layout. Required: maximum absolute difference exactly `0.0`.
3. The test has power: a cold-start stream (no priming) on the same rows **does** differ.
4. The feature table is cached as `data/cache/features_v<version>.parquet`.

**Design choices (viva-ready)**

| Choice | Why |
|---|---|
| **No label is read.** `isFraud` and `isFlaggedFraud` are dropped right after loading. | Nothing decided here can be informed by labels, so this notebook cannot leak them. The feature function never sees them either (D2). |
| **Exact equality, not a tolerance.** | Both paths run the same Python functions on the same floats in the same order, so any non-zero difference is a real bug, not rounding. A tolerance would hide it. |
| **Parity runs the worker sequence with a JSON-backed store.** | `process_event` is used by training; the worker calls `compute_features` and `update_dest_state` separately and keeps state in Redis. Using the separate calls plus a JSON round trip tests the code path and the serialisation the worker will use, without needing Redis yet. |
| **A cold-start control run.** | A test that cannot fail proves nothing. If the cold start were identical to the batch, the priming step would be untested. |
| **Memory is sampled, not estimated.** | A background thread records the process RSS every 50 ms. Short spikes between samples can be missed, so treat the figure as a close lower bound on the true peak. |

**How to run:** restart the kernel, run all cells. Run it **twice**; the printed features hash must be identical. The pass over all rows takes a while (every event is validated by Pydantic, as the real ingress does), and parity needs one more pass over about 95% of the rows. The notebook stops with an error at the end if any hard gate fails.

In [1]:
from data_loader import load_paysim
raw = load_paysim()

In [ ]:
# ---- Pre-registered constants: fixed BEFORE the first run, committed unchanged. ----
EXPECTED_SCHEMA_VERSION = "1.0.0"   # becomes "1.0.0" in the freeze commit (FEATURE_SPEC section 9)
EXPECTED_N_FEATURES = 12
EXPECTED_N_ROWS = 6_362_620         # from the data_loader.py run on the real CSV
SAMPLE_ROWS = 1_000_000             # memory/time measurement on the first N rows
PARITY_MAX_ABS_DIFF = 0.0           # exact equality, not a tolerance
VALIDATE = True                     # Pydantic-validate every event, as stream/API/upload ingress does
RUN_1M_MEASUREMENT = True           # False for a cleaner full-pass-only memory number (restart kernel first)
REPLAY_START_OVERRIDE = None        # int; only if docs/split_boundaries.json cannot be parsed (cell 4)
REPLAY_SHARE_RANGE = (0.03, 0.07)   # sanity bound on the replay boundary (README 7.1: about 5%)
MEM_SAMPLE_INTERVAL_S = 0.05
SPLIT_FILE = "docs/split_boundaries.json"
RESULTS_FILE = "docs/results/s1_09_feature_run.json"

In [3]:
import gc
import hashlib
import json
import os
import platform
import sys
import threading
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import psutil
except ImportError as exc:  # not necessarily in requirements.txt
    raise SystemExit("This notebook needs psutil for memory sampling: pip install psutil") from exc


def find_repo_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "shared" / "features").is_dir() and (p / "docs").is_dir():
            return p
    raise RuntimeError("repo root not found (expected shared/features/ and docs/ above the notebook)")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)  # data_loader.py uses relative paths (data/raw, data/cache)
for p in (REPO_ROOT, REPO_ROOT / "notebooks"):  # data_loader.py may sit in either
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from data_loader import load_paysim
from shared.features.batch import build_feature_table
from shared.features.core import (
    CONFIG,
    EMPTY_STATE,
    FEATURE_NAMES,
    FEATURE_SCHEMA_VERSION,
    PAYSIM_COLUMN_MAP,
    SCORED_TYPES,
    AccountState,
    TransactionEvent,
    compute_features,
    update_dest_state,
)

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 200)
print("repo root:", REPO_ROOT)
print("python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__,
      "| platform", platform.platform())

GATES: list[dict] = []


def gate(gid: str, question: str, result: str, detail: str) -> bool:
    """result: PASS / FAIL / SKIP / INFO. Hard gates are every PASS/FAIL/SKIP row; INFO rows are reported only."""
    assert result in {"PASS", "FAIL", "SKIP", "INFO"}
    GATES.append({"gate": gid, "question": question, "result": result, "detail": detail})
    print(f"[{result}] {gid}: {detail}")
    return result == "PASS"

repo root: D:\Projects\fraud_detection
python 3.11.5 | pandas 3.0.5 | numpy 2.4.6 | platform Windows-10-10.0.26300-SP0


## 1. Load the data (labels dropped immediately)

In [4]:
t0 = time.perf_counter()
_raw = load_paysim()  # Parquet cache; original CSV row order
df = _raw[list(PAYSIM_COLUMN_MAP)].copy()  # only the nine columns the feature function maps; labels dropped here
del _raw
gc.collect()
print(f"loaded {df.shape} in {time.perf_counter() - t0:.1f}s")
assert df.index.equals(pd.RangeIndex(len(df))), "row index must be 0..n-1 in original CSV order"
assert df["step"].is_monotonic_increasing, "gate G3 prerequisite: rows must be in step order"
SCORED_LIST = sorted(SCORED_TYPES)
N_ROWS = len(df)
N_SCORED = int(df["type"].isin(SCORED_LIST).sum())
print("rows:", N_ROWS, "| scored rows (TRANSFER + CASH_OUT):", N_SCORED)
print(df.dtypes.to_string())

gate("F1", "Schema and feature list as pre-registered?",
     "PASS" if (FEATURE_SCHEMA_VERSION == EXPECTED_SCHEMA_VERSION
                and len(FEATURE_NAMES) == EXPECTED_N_FEATURES
                and not any("orig_inbound" in n for n in FEATURE_NAMES)) else "FAIL",
     f"schema {FEATURE_SCHEMA_VERSION}, {len(FEATURE_NAMES)} features, no origin-side chain features")
gate("F2", "Row count equals the data_loader.py run on the real CSV?",
     "PASS" if N_ROWS == EXPECTED_N_ROWS else "FAIL", f"{N_ROWS:,} rows (expected {EXPECTED_N_ROWS:,})")

loaded (6362620, 9) in 0.6s
rows: 6362620 | scored rows (TRANSFER + CASH_OUT): 2770409
step                 int64
type              category
amount             float64
nameOrig               str
oldbalanceOrg      float64
newbalanceOrig     float64
nameDest               str
oldbalanceDest     float64
newbalanceDest     float64
[PASS] F1: schema 0.2.0, 12 features, no origin-side chain features
[PASS] F2: 6,362,620 rows (expected 6,362,620)


True

## 2. Replay-pool boundary from `docs/split_boundaries.json`

The first row of the replay pool (a row position in the original order, README 7.1). The file's layout is read defensively and **sanity-checked against the data** (about 5% of rows, cut at the start of a `step`), because the parser was written without seeing the file.

In [5]:
def find_replay_start(obj) -> tuple[int, str]:
    START_KEYS = ("start", "start_row", "start_idx", "start_index", "first_row", "begin", "from", "row_start")
    found: list[tuple[str, int]] = []

    def is_int(v):
        return isinstance(v, int) and not isinstance(v, bool)

    def from_dict(d: dict, where: str):
        for k in START_KEYS:
            if k in d and is_int(d[k]):
                found.append((f"{where}.{k}", d[k]))

    def walk(node, path: str):
        if isinstance(node, dict):
            for k, v in node.items():
                here = f"{path}.{k}" if path else str(k)
                if "replay" in str(k).lower():
                    if isinstance(v, dict):
                        from_dict(v, here)
                    elif isinstance(v, list) and len(v) == 2 and all(is_int(x) for x in v):
                        found.append((f"{here}[0]", v[0]))
                    elif is_int(v) and any(w in str(k).lower() for w in ("start", "first", "begin")):
                        found.append((here, v))
                walk(v, here)
            if any("replay" in str(v).lower() for k, v in node.items() if k in ("name", "split") and isinstance(v, str)):
                from_dict(node, path)
        elif isinstance(node, list):
            for i, v in enumerate(node):
                walk(v, f"{path}[{i}]")

    walk(obj, "")
    values = {v for _, v in found}
    if len(values) != 1:
        raise ValueError(
            "Could not identify one replay-start row in " + SPLIT_FILE + f" (candidates: {found}). "
            "Set REPLAY_START_OVERRIDE in the config cell. File content:\n" + json.dumps(obj, indent=2)[:3000])
    return values.pop(), found[0][0]


if REPLAY_START_OVERRIDE is not None:
    R0, R0_SOURCE = int(REPLAY_START_OVERRIDE), "REPLAY_START_OVERRIDE"
else:
    with open(SPLIT_FILE, encoding="utf-8") as fh:
        SPLIT_JSON = json.load(fh)
    R0, R0_SOURCE = find_replay_start(SPLIT_JSON)

share = 1 - R0 / N_ROWS
step_snapped = bool(df["step"].iat[R0 - 1] < df["step"].iat[R0])
print(f"replay start row {R0:,} (from {R0_SOURCE}); replay share {share:.4f}; snapped to a step start: {step_snapped}")
assert REPLAY_SHARE_RANGE[0] <= share <= REPLAY_SHARE_RANGE[1], "replay share outside the sanity range - check the split file"
assert step_snapped, "the replay boundary must fall on the first row of a step (FEATURE_SPEC section 7)"
TAIL_ROWS = N_ROWS - R0

replay start row 6,040,750 (from splits.replay.row_start); replay share 0.0506; snapped to a step start: True


## 3. Helpers (comparison and measurement)

In [6]:
def max_abs_diff(a: pd.DataFrame, b: pd.DataFrame) -> float:
    """0.0 only if both frames are non-empty, have identical index and columns, are finite, and are equal element for element."""
    if len(a) == 0 or len(b) == 0:
        return float("inf")
    if not a.index.equals(b.index) or list(a.columns) != list(b.columns):
        return float("inf")
    av, bv = a.to_numpy(dtype="float64"), b.to_numpy(dtype="float64")
    if np.isnan(av).any() or np.isnan(bv).any():
        return float("inf")
    return float(np.abs(av - bv).max())


def lifetime_peak_rss_mb():
    """Process-lifetime peak (includes loading the data). Windows: peak working set; Linux/macOS: ru_maxrss."""
    mi = psutil.Process().memory_info()
    if hasattr(mi, "peak_wset"):
        return mi.peak_wset / 2**20
    try:
        import resource
        r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        return (r if sys.platform == "darwin" else r * 1024) / 2**20
    except Exception:
        return None


class PeakRSS:
    """Samples process RSS in a background thread while the block runs."""

    def __init__(self, interval: float):
        self.interval = interval

    def __enter__(self):
        self._proc = psutil.Process()
        self.start = self.peak = self._proc.memory_info().rss
        self._stop = threading.Event()
        self._t = threading.Thread(target=self._run, daemon=True)
        self._t.start()
        return self

    def _run(self):
        while not self._stop.is_set():
            self.peak = max(self.peak, self._proc.memory_info().rss)
            self._stop.wait(self.interval)

    def __exit__(self, *exc):
        self._stop.set()
        self._t.join()
        self.end = self._proc.memory_info().rss
        self.peak = max(self.peak, self.end)
        return False


def measured_pass(label: str, frame: pd.DataFrame, validate: bool):
    gc.collect()
    with PeakRSS(MEM_SAMPLE_INTERVAL_S) as mem:
        t0 = time.perf_counter()
        feats_, store_ = build_feature_table(frame, validate=validate)
        secs = time.perf_counter() - t0
    mb = 2**20
    rec = {
        "label": label, "validate": validate, "rows": len(frame), "scored_rows": len(feats_),
        "seconds": round(secs, 1), "rows_per_second": round(len(frame) / secs),
        "rss_before_mb": round(mem.start / mb), "rss_peak_sampled_mb": round(mem.peak / mb),
        "rss_growth_mb": round((mem.peak - mem.start) / mb),
        "lifetime_peak_rss_mb": None if lifetime_peak_rss_mb() is None else round(lifetime_peak_rss_mb()),
        "state_accounts": len(store_.data),
    }
    print(rec)
    return feats_, store_, rec

## 4. First 1,000,000 rows: time and peak memory

Run first, on a freshly loaded frame. Also checks that the trusted path (`validate=False`, `model_construct`) gives identical features to the validated path, because training may use the fast path while the stream always validates.

In [7]:
REC_1M = None
F_1M = None
NOVALIDATE_DIFF = None
if RUN_1M_MEASUREMENT:
    head_1m = df.iloc[:SAMPLE_ROWS]
    F_1M, store_1m, REC_1M = measured_pass("first_1M", head_1m, VALIDATE)
    del store_1m
    gc.collect()
    F_nv, store_nv = build_feature_table(head_1m, validate=False)
    NOVALIDATE_DIFF = max_abs_diff(F_1M, F_nv)
    del F_nv, store_nv
    gc.collect()
    print("validate=True vs validate=False, max abs diff:", NOVALIDATE_DIFF)
    gate("F5", "Trusted path (validate=False) equals the validated path on the first 1M rows?",
         "PASS" if NOVALIDATE_DIFF == PARITY_MAX_ABS_DIFF else "FAIL", f"max abs diff {NOVALIDATE_DIFF}")
else:
    gate("F5", "Trusted path (validate=False) equals the validated path on the first 1M rows?", "SKIP",
         "RUN_1M_MEASUREMENT is False")

{'label': 'first_1M', 'validate': True, 'rows': 1000000, 'scored_rows': 445100, 'seconds': 8.2, 'rows_per_second': 122670, 'rss_before_mb': 1950, 'rss_peak_sampled_mb': 1950, 'rss_growth_mb': 0, 'lifetime_peak_rss_mb': 2009, 'state_accounts': 38876}
validate=True vs validate=False, max abs diff: 0.0
[PASS] F5: max abs diff 0.0


## 5. Full pass over all rows, integrity checks, and the cache

One chronological pass over every row (D5): state carries across the split boundaries; each row's features still use only earlier rows.

In [8]:
FEATS, store_full, REC_FULL = measured_pass("full", df, VALIDATE)
N_DISTINCT_STATE_ACCOUNTS = len(store_full.data)
del store_full
gc.collect()

scored_positions = np.flatnonzero(df["type"].isin(SCORED_LIST).to_numpy())
flag_cols = ["type_is_transfer", "orig_balance_zero", "orig_drained", "dest_old_balance_zero",
             "dest_new_balance_zero", "dest_has_prior_inbound_transfer"]
cnt = FEATS["dest_inbound_transfer_count_24h"]
checks = {
    "columns equal FEATURE_NAMES in order": list(FEATS.columns) == list(FEATURE_NAMES),
    "one row per scored row": len(FEATS) == N_SCORED,
    "index = original positions of scored rows": FEATS.index.equals(pd.Index(scored_positions)),
    "all values finite": bool(np.isfinite(FEATS.to_numpy()).all()),
    "ratio within [0, clip]": bool(FEATS["amount_to_orig_balance"].between(0, CONFIG.ratio_clip).all()),
    "flags are 0/1": all(set(FEATS[c].unique()) <= {0.0, 1.0} for c in flag_cols),
    "count is a non-negative integer": bool(((cnt >= 0) & (cnt == np.floor(cnt))).all()),
    "log features non-negative": bool((FEATS["log_amount"] >= 0).all() and (FEATS["log_orig_balance_before"] >= 0).all()),
}
print(pd.Series(checks).to_string())
gate("F3", "Feature table well-formed (columns, rows, finite, ranges)?",
     "PASS" if all(checks.values()) else "FAIL",
     f"{len(FEATS):,} rows x {FEATS.shape[1]} columns; failed: {[k for k, v in checks.items() if not v] or 'none'}")

if F_1M is not None:
    prefix_1m = max_abs_diff(F_1M, FEATS.loc[F_1M.index])
    gate("F4a", "First-1M features equal the same rows from the full pass (no future leakage, FEATURE_SPEC test 3 on real data)?",
         "PASS" if prefix_1m == 0.0 else "FAIL", f"max abs diff {prefix_1m}")
else:
    gate("F4a", "First-1M features equal the same rows from the full pass?", "SKIP", "RUN_1M_MEASUREMENT is False")
del F_1M
gc.collect()

# ---- cache ----
CACHE_PATH = Path("data/cache") / f"features_v{FEATURE_SCHEMA_VERSION}.parquet"
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
FEATS.to_parquet(CACHE_PATH, engine="pyarrow")  # gitignored
_back = pd.read_parquet(CACHE_PATH)
cache_diff = max_abs_diff(FEATS, _back)
del _back
gate("F6", f"Cache {CACHE_PATH.as_posix()} reads back identical?", "PASS" if cache_diff == 0.0 else "FAIL",
     f"max abs diff {cache_diff}; {CACHE_PATH.stat().st_size / 2**20:.0f} MB on disk")

_h = hashlib.sha256()
_h.update(np.ascontiguousarray(FEATS.index.to_numpy(dtype="int64")).tobytes())
_h.update(np.ascontiguousarray(FEATS.to_numpy(dtype="float64")).tobytes())
FEATURES_SHA256 = _h.hexdigest()
print("features sha256 (index + values):", FEATURES_SHA256)

{'label': 'full', 'validate': True, 'rows': 6362620, 'scored_rows': 2770409, 'seconds': 79.7, 'rows_per_second': 79875, 'rss_before_mb': 1332, 'rss_peak_sampled_mb': 3047, 'rss_growth_mb': 1715, 'lifetime_peak_rss_mb': 3113, 'state_accounts': 237006}
columns equal FEATURE_NAMES in order         True
one row per scored row                       True
index = original positions of scored rows    True
all values finite                            True
ratio within [0, clip]                       True
flags are 0/1                                True
count is a non-negative integer              True
log features non-negative                    True
[PASS] F3: 2,770,409 rows x 12 columns; failed: none
[PASS] F4a: max abs diff 0.0
[PASS] F6: max abs diff 0.0; 81 MB on disk
features sha256 (index + values): 5f102f58ba3618731f95a07f6834a068e618736a9f274e26ef52156ca83adea9


## 6. Priming parity on the real replay pool (D6)

1. Batch pass over the rows **before** the replay pool -> state at the boundary.
2. Copy that state, for the recipient accounts of the replay pool's scored rows only, into a JSON-backed store (the planned Redis layout: `{"inbound": [[step, amount], ...], "total": n}`).
3. Stream the replay pool through the **worker sequence**: read recipient state -> `compute_features` -> `update_dest_state` -> write state.
4. Compare with the batch features for the same rows. Required: maximum absolute difference exactly `0.0`.
5. Control: the same stream from an empty store must differ.

In [9]:
class JsonStore:
    """State store that round-trips every read/write through JSON, as the Redis layout in FEATURE_SPEC section 5 will."""

    def __init__(self):
        self.raw: dict[str, str] = {}

    def get(self, name: str) -> AccountState:
        s = self.raw.get(name)
        if s is None:
            return EMPTY_STATE
        d = json.loads(s)
        return AccountState(inbound=tuple((int(a), float(b)) for a, b in d["inbound"]), inbound_total=int(d["total"]))

    def put(self, name: str, state: AccountState) -> None:
        self.raw[name] = json.dumps({"inbound": [[s, a] for s, a in state.inbound], "total": state.inbound_total})


SRC = list(PAYSIM_COLUMN_MAP)
DST = [PAYSIM_COLUMN_MAP[c] for c in SRC]


def stream_replay(frame: pd.DataFrame, store: JsonStore) -> pd.DataFrame:
    """Flow A worker sequence (FEATURE_SPEC section 5): read state, compute, then update state. Events are validated."""
    idxs, rows = [], []
    for idx, *vals in frame[SRC].itertuples(name=None):
        e = TransactionEvent(**dict(zip(DST, vals)))
        dest = store.get(e.name_dest)
        if e.type in SCORED_TYPES:
            idxs.append(idx)
            rows.append(list(compute_features(e, dest).values()))
        new = update_dest_state(e, dest)
        if new is not dest:
            store.put(e.name_dest, new)
    return pd.DataFrame(rows, index=pd.Index(idxs), columns=list(FEATURE_NAMES))


head, tail = df.iloc[:R0], df.iloc[R0:]

# 1. batch pass over the head only
t0 = time.perf_counter()
F_HEAD, store_head = build_feature_table(head, validate=VALIDATE)
print(f"head pass: {len(head):,} rows in {time.perf_counter() - t0:.0f}s, {len(store_head.data):,} state accounts")
prefix_head = max_abs_diff(F_HEAD, FEATS.loc[F_HEAD.index])
gate("F4b", "Head-only features equal the same rows from the full pass (prefix property, real data)?",
     "PASS" if prefix_head == 0.0 else "FAIL", f"max abs diff {prefix_head}")
del F_HEAD

# 2. prime: recipients of the replay pool's scored rows only
recipients = set(tail.loc[tail["type"].isin(SCORED_LIST), "nameDest"])
primed = JsonStore()
n_primed = 0
for name in recipients:
    if name in store_head.data:  # accounts never seen before the boundary stay absent, as in Redis
        primed.put(name, store_head.data[name])
        n_primed += 1
print(f"replay recipients: {len(recipients):,}; with prior state, primed: {n_primed:,}")
del store_head
gc.collect()

# 3-4. stream with priming and compare
t0 = time.perf_counter()
STREAM_PRIMED = stream_replay(tail, primed)
print(f"primed stream: {len(tail):,} rows in {time.perf_counter() - t0:.0f}s")
BATCH_TAIL = FEATS.loc[FEATS.index >= R0]
PARITY_DIFF = max_abs_diff(STREAM_PRIMED, BATCH_TAIL)
bitwise_equal = bool(len(STREAM_PRIMED) == len(BATCH_TAIL) and (STREAM_PRIMED.to_numpy() == BATCH_TAIL.to_numpy()).all())
print("parity max abs diff:", PARITY_DIFF, "| bitwise equal:", bitwise_equal)
gate("F7", "Primed stream features equal the batch features on the real replay pool (exactly)?",
     "PASS" if (PARITY_DIFF <= PARITY_MAX_ABS_DIFF and bitwise_equal) else "FAIL",
     f"{len(BATCH_TAIL):,} scored replay rows; max abs diff {PARITY_DIFF}; primed accounts {n_primed:,}")

# 5. control: cold start
STREAM_COLD = stream_replay(tail, JsonStore())
if STREAM_COLD.index.equals(BATCH_TAIL.index):
    differs = (STREAM_COLD.to_numpy() != BATCH_TAIL.to_numpy())
    COLD_ROWS_DIFFERING = int(differs.any(axis=1).sum())
    COLD_BY_COLUMN = {c: int(differs[:, i].sum()) for i, c in enumerate(FEATURE_NAMES) if differs[:, i].any()}
else:
    COLD_ROWS_DIFFERING, COLD_BY_COLUMN = -1, {}
COLD_DIFF = max_abs_diff(STREAM_COLD, BATCH_TAIL)
print("cold start: rows differing", COLD_ROWS_DIFFERING, "| by column", COLD_BY_COLUMN, "| max abs diff", COLD_DIFF)
gate("F8", "Control: does an unprimed (cold-start) stream differ, so the parity test has power?",
     "PASS" if COLD_ROWS_DIFFERING > 0 else "FAIL",
     f"{COLD_ROWS_DIFFERING:,} of {len(BATCH_TAIL):,} rows differ, columns {sorted(COLD_BY_COLUMN)}")

head pass: 6,040,750 rows in 70s, 222,621 state accounts
[PASS] F4b: max abs diff 0.0
replay recipients: 119,253; with prior state, primed: 50,325
primed stream: 321,870 rows in 3s
parity max abs diff: 0.0 | bitwise equal: True
[PASS] F7: 133,794 scored replay rows; max abs diff 0.0; primed accounts 50,325
cold start: rows differing 55474 | by column {'dest_inbound_transfer_count_24h': 24, 'dest_has_prior_inbound_transfer': 55474} | max abs diff 1.0
[PASS] F8: 55,474 of 133,794 rows differ, columns ['dest_has_prior_inbound_transfer', 'dest_inbound_transfer_count_24h']


True

## 7. Gate summary and run report

In [10]:
summary = pd.DataFrame(GATES)
print(summary[["gate", "result", "detail"]].to_string(index=False))

cost = pd.DataFrame([r for r in (REC_1M, REC_FULL) if r is not None])
print("\nTime and memory (RSS sampled every %.0f ms; rss_before includes the loaded frame; growth = extra memory the pass itself needed):" % (MEM_SAMPLE_INTERVAL_S * 1000))
print(cost.to_string(index=False))
print("\nstate accounts after the full pass:", f"{N_DISTINCT_STATE_ACCOUNTS:,}")

bad = [g["gate"] for g in GATES if g["result"] in ("FAIL", "SKIP")]
READY_TO_FREEZE = not bad
print("\nREADY TO FREEZE AT 1.0.0:", READY_TO_FREEZE, "" if READY_TO_FREEZE else f"(not ready: {bad})")

report = {
    "run_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook": "notebooks/02_features_and_parity.ipynb",
    "feature_schema_version": FEATURE_SCHEMA_VERSION,
    "environment": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
                    "platform": platform.platform()},
    "config": {"validate": VALIDATE, "sample_rows": SAMPLE_ROWS, "window_steps": CONFIG.window_steps,
               "ratio_clip": CONFIG.ratio_clip},
    "data": {"rows": N_ROWS, "scored_rows": N_SCORED, "replay_start_row": R0, "replay_start_source": R0_SOURCE,
             "replay_rows": TAIL_ROWS, "replay_scored_rows": len(BATCH_TAIL)},
    "cost": {"first_1M": REC_1M, "full": REC_FULL, "state_accounts_after_full_pass": N_DISTINCT_STATE_ACCOUNTS},
    "parity": {"max_abs_diff": PARITY_DIFF, "bitwise_equal": bitwise_equal, "primed_accounts": n_primed,
               "replay_recipients": len(recipients), "prefix_head_diff": prefix_head,
               "validate_false_diff": NOVALIDATE_DIFF, "cold_start_rows_differing": COLD_ROWS_DIFFERING,
               "cold_start_by_column": COLD_BY_COLUMN},
    "features_sha256": FEATURES_SHA256,
    "cache_file": CACHE_PATH.as_posix(),
    "gates": GATES,
    "ready_to_freeze": READY_TO_FREEZE,
}
Path(RESULTS_FILE).parent.mkdir(parents=True, exist_ok=True)
Path(RESULTS_FILE).write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")
print("report written:", RESULTS_FILE)

if bad:
    raise AssertionError(f"S1-09 gates not all PASS: {bad}")

gate result                                                                                                        detail
  F1   PASS                                                      schema 0.2.0, 12 features, no origin-side chain features
  F2   PASS                                                                           6,362,620 rows (expected 6,362,620)
  F5   PASS                                                                                              max abs diff 0.0
  F3   PASS                                                                     2,770,409 rows x 12 columns; failed: none
 F4a   PASS                                                                                              max abs diff 0.0
  F6   PASS                                                                               max abs diff 0.0; 81 MB on disk
 F4b   PASS                                                                                              max abs diff 0.0
  F7   PASS             

## 8. If everything passed

1. Run the notebook a **second time** (restart kernel, run all). The features hash must be identical to the first run.
2. Paste the numbers from `docs/results/s1_09_feature_run.json` into `docs/FEATURE_SPEC.md` section 6.
3. Freeze: see "Freezing at 1.0.0" in the section 6 additions. After the version bump, re-run this notebook with `EXPECTED_SCHEMA_VERSION = "1.0.0"`; the features hash must equal the `0.2.0` hash (nothing but the label changed).
4. Still open from S1-09: repeat the parity test against the **real Redis-backed store** once it exists. The JSON store here stands in for it.